In [1]:
# 基于迁移学习的两阶段方法

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import copy
import os
import pandas as pd
from sklearn.metrics import mean_squared_error
import torch.nn.functional as F  # 添加F导入，用于calculate_transfer_metrics
import logging  # 添加logging导入

# 配置日志
logging.basicConfig(level=logging.INFO)

# 导入模型
from AdaptiveBILSTM import AdaptiveBiLSTM, DomainDiscriminator, MultiScaleEncoder

# 修改：导入新的数据加载器函数
from dataloader import (
    load_source_domain_dataloaders,
    load_transfer_learning_dataloaders
)

# 导入计算函数
from calculate import calculate_metrics, print_metrics_table, calculate_uncertainty_metrics

# 设置随机种子以确保可重复性
torch.manual_seed(42)
np.random.seed(42)

# 检查CUDA是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

使用设备: cuda


In [2]:
# 模型参数设置
batch_size = 32
sequence_length = 24
forecast_horizon = 24
hidden_dim = 64
num_layers = 2
dropout = 0.2
learning_rate = 0.001
weight_decay = 1e-4
epochs = 10  # 源域预训练轮次

# 修改：输入维度从原来可能不同的值改为6（1个电力特征 + 5个天气特征）
input_dim = 6  # 1个电力 + 5个天气特征（气温、露点温度、气压、风向、风速）

# 修改：域的数量改为类别数量
num_categories = 6  # DO, HO, LI, OF, UL, CC
num_domains = num_categories  # 使用类别作为域

In [3]:

import torch
import numpy as np

def predict_with_uncertainty(model, inputs, category_onehot, domain_idx=None, mc_samples=100, device='cuda'):
    """
    使用MC Dropout进行不确定性估计 - 适配新的数据格式
    
    Args:
        model: 模型
        inputs: 输入特征 [batch, buildings, seq_len, features]
        category_onehot: 类别的one-hot编码 [batch, num_categories]
        domain_idx: 域索引（可选）
        mc_samples: MC采样次数
        device: 设备
    """
    model = model.to(device)
    inputs = inputs.to(device)
    category_onehot = category_onehot.to(device)  # 修改：从category改为category_onehot
    model.train()  # 开启dropout以进行随机采样
    
    predictions = []
    for _ in range(mc_samples):
        # 修改: 传入category_onehot而不是category
        outputs = model(inputs, category_onehot, domain_idx=domain_idx)
        
        if isinstance(outputs, tuple):
            outputs = outputs[0]  # 如果模型返回元组，取第一个元素
        
        predictions.append(outputs.detach())
    
    # 将预测堆叠为形状[mc_samples, batch, buildings, forecasts]
    try:
        stacked_preds = torch.stack(predictions, dim=0)
        
        # 计算平均值和标准差
        mean_pred = torch.mean(stacked_preds, dim=0)
        std_pred = torch.std(stacked_preds, dim=0)
        
        # 计算95%置信区间
        lower_bound = mean_pred - 1.96 * std_pred
        upper_bound = mean_pred + 1.96 * std_pred
        
        # 确保边界在有效范围内（注意：这里可能需要根据你的数据范围调整）
        # 如果数据已经归一化到[0,1]，保持不变
        # 如果没有归一化，需要调整为实际的数据范围
        lower_bound = torch.clamp(lower_bound, 0, float('inf'))  # 修改：电力消耗不能为负
        upper_bound = torch.clamp(upper_bound, 0, float('inf'))
        
        # 转换为CPU NumPy数组
        return (mean_pred.cpu().numpy(),
                lower_bound.cpu().numpy(),
                upper_bound.cpu().numpy(),
                std_pred.cpu().numpy())
    
    except Exception as e:
        print(f"处理MC采样结果时出错: {str(e)}")
        # 如果堆叠或其他操作失败，使用第一个样本作为预测
        try:
            mean_pred = predictions[0].cpu().numpy()
            std_pred = np.ones_like(mean_pred) * 0.1
            lower_bound = np.maximum(mean_pred - 1.96 * std_pred, 0)
            upper_bound = mean_pred + 1.96 * std_pred
            return mean_pred, lower_bound, upper_bound, std_pred
        except:
            # 最后的后备方案：返回零数组
            # 修改：根据输入形状推断输出形状
            if len(inputs.shape) == 4:  # [batch, buildings, seq_len, features]
                batch_size = inputs.shape[0]
                num_buildings = inputs.shape[1]
                output_shape = (batch_size, num_buildings, forecast_horizon)
            else:
                output_shape = (inputs.shape[0], forecast_horizon)
            
            zeros = np.zeros(output_shape)
            return zeros, zeros, zeros, zeros
def calculate_transfer_metrics(source_features, target_features, source_outputs=None, 
                              target_outputs=None, baseline_predictions=None, targets=None):
    """
    计算迁移学习的综合评估指标
    
    Args:
        source_features: 源域特征 [batch_size, feature_dim] 或 [batch_size, seq_len, feature_dim]
        target_features: 目标域特征 [batch_size, feature_dim] 或 [batch_size, seq_len, feature_dim]
        source_outputs: 源域模型输出（可选）
        target_outputs: 目标域模型输出（可选）
        baseline_predictions: 基线模型预测（可选）
        targets: 真实目标值（可选，用于计算负迁移）
    """
    import traceback  # 导入traceback以便打印详细错误信息
    
    # 处理输入特征的维度
    # 新的数据格式可能是 [batch, buildings, seq_len, features]
    if len(source_features.shape) == 4:  # [batch, buildings, seq_len, features]
        logging.info(f"检测到4D特征，进行展平: source_features.shape={source_features.shape}")
        # 先将buildings维度和batch维度合并，然后平均时间步
        batch_size, num_buildings, seq_len, feature_dim = source_features.shape
        source_features = source_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
        source_features = source_features.mean(dim=1)  # [batch*buildings, feature_dim]
        
        # 对目标特征做同样处理
        target_features = target_features.reshape(
            target_features.shape[0] * target_features.shape[1], 
            target_features.shape[2], 
            target_features.shape[3]
        )
        target_features = target_features.mean(dim=1)
        
        logging.info(f"展平后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    elif len(source_features.shape) == 3:  # [batch, seq_len, feature_dim]
        logging.info(f"检测到3D特征，进行展平: source_features.shape={source_features.shape}")
        # 平均所有时间步
        source_features = source_features.mean(dim=1)  # [batch, feature_dim]
        target_features = target_features.mean(dim=1)  # [batch, feature_dim]
        
        logging.info(f"展平后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    # 确保特征维度匹配
    if source_features.size(-1) != target_features.size(-1):  # 使用-1获取最后一个维度
        logging.warning(f"特征维度不匹配: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
        return {
            'a_distance': 'N/A',
            'feature_alignment': 'N/A',
            'mmd': 'N/A',
            'sample_efficiency': None
        }
    
    # 检查特征数量是否过大，可能导致内存问题
    max_samples = 5000
    if source_features.size(0) > max_samples or target_features.size(0) > max_samples:
        logging.warning(f"特征数量过大，进行采样: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
        
        if source_features.size(0) > max_samples:
            indices = torch.randperm(source_features.size(0))[:max_samples]
            source_features = source_features[indices]
        
        if target_features.size(0) > max_samples:
            indices = torch.randperm(target_features.size(0))[:max_samples]
            target_features = target_features[indices]
        
        logging.info(f"采样后: source_features.shape={source_features.shape}, target_features.shape={target_features.shape}")
    
    # 下面的辅助函数保持不变
    def calculate_mmd(x, y):
        """计算最大平均差异(MMD)"""
        try:
            x_kernel = torch.mm(x, x.t())
            y_kernel = torch.mm(y, y.t())
            xy_kernel = torch.mm(x, y.t())
            return x_kernel.mean() + y_kernel.mean() - 2 * xy_kernel.mean()
        except Exception as e:
            logging.error(f"计算MMD时出错: {str(e)}")
            return float('nan')
    
    def calculate_a_distance(source_features, target_features):
        """计算A-distance"""
        try:
            domain_classifier = nn.Sequential(
                nn.Linear(source_features.size(1), 50),
                nn.ReLU(),
                nn.Linear(50, 1)
            ).to(source_features.device)
            
            source_domain_labels = torch.ones(source_features.size(0), 1).to(source_features.device)
            target_domain_labels = torch.zeros(target_features.size(0), 1).to(source_features.device)
            
            features = torch.cat([source_features, target_features], dim=0)
            labels = torch.cat([source_domain_labels, target_domain_labels], dim=0)
            
            optimizer = torch.optim.Adam(domain_classifier.parameters())
            criterion = nn.BCEWithLogitsLoss()
            
            for _ in range(100):
                optimizer.zero_grad()
                preds = domain_classifier(features)
                loss = criterion(preds, labels)
                loss.backward()
                optimizer.step()
            
            with torch.no_grad():
                preds = torch.sigmoid(domain_classifier(features))
                predicted_labels = (preds > 0.5).float()
                error = (predicted_labels != labels).float().mean()
            
            return 2 * (1 - 2 * error)
        except Exception as e:
            logging.error(f"计算A-distance时出错: {str(e)}")
            traceback.print_exc()
            return float('nan')
    
    def calculate_feature_alignment(source_features, target_features):
        """计算特征对齐质量"""
        try:
            source_norm = F.normalize(source_features, p=2, dim=1)
            target_norm = F.normalize(target_features, p=2, dim=1)
            similarity = torch.mm(source_norm, target_norm.t())
            return similarity.mean()
        except Exception as e:
            logging.error(f"计算特征对齐质量时出错: {str(e)}")
            return float('nan')
    
    def detect_negative_transfer(target_loss, baseline_loss):
        """检测负迁移"""
        if isinstance(target_loss, torch.Tensor):
            target_loss = target_loss.item()
        if isinstance(baseline_loss, torch.Tensor):
            baseline_loss = baseline_loss.item()
            
        is_negative = target_loss > baseline_loss
        transfer_gain = baseline_loss - target_loss
        
        return {
            'is_negative': bool(is_negative),
            'transfer_gain': float(transfer_gain)
        }
    
    def calculate_sample_efficiency(performance_curve):
        """计算样本效率"""
        target_performance = 0.9
        for i, perf in enumerate(performance_curve):
            if perf >= target_performance:
                return i + 1
        return len(performance_curve)

    # 计算基本指标
    try:
        metrics = {
            'a_distance': calculate_a_distance(source_features, target_features),
            'feature_alignment': calculate_feature_alignment(source_features, target_features),
            'mmd': calculate_mmd(source_features, target_features),
            'sample_efficiency': None
        }
    except Exception as e:
        logging.error(f"计算迁移学习指标时出错: {str(e)}")
        traceback.print_exc()
        return {
            'a_distance': 'N/A',
            'feature_alignment': 'N/A',
            'mmd': 'N/A',
            'sample_efficiency': None
        }
    
    # 如果提供了输出、基线预测和目标值，检测负迁移
    if target_outputs is not None and baseline_predictions is not None and targets is not None:
        try:
            # 处理可能的多维数据
            if isinstance(target_outputs, torch.Tensor):
                target_outputs = target_outputs.detach().cpu().numpy()
            if isinstance(baseline_predictions, torch.Tensor):
                baseline_predictions = baseline_predictions.detach().cpu().numpy()
            if isinstance(targets, torch.Tensor):
                targets = targets.detach().cpu().numpy()
            
            # 确保都是一维数组
            target_outputs = np.array(target_outputs).flatten()
            baseline_predictions = np.array(baseline_predictions).flatten()
            targets = np.array(targets).flatten()
            
            # 确保所有数组长度相同
            min_length = min(len(target_outputs), len(baseline_predictions), len(targets))
            target_outputs = target_outputs[:min_length]
            baseline_predictions = baseline_predictions[:min_length]
            targets = targets[:min_length]
            
            # 计算MSE损失
            target_loss = np.mean((target_outputs - targets) ** 2)
            baseline_loss = np.mean((baseline_predictions - targets) ** 2)
            
            logging.debug(f"target_loss: {target_loss}, type: {type(target_loss)}")
            logging.debug(f"baseline_loss: {baseline_loss}, type: {type(baseline_loss)}")
            
            # 检测负迁移
            neg_transfer = detect_negative_transfer(target_loss, baseline_loss)
            metrics['is_negative_transfer'] = neg_transfer['is_negative']
            metrics['transfer_gain'] = neg_transfer['transfer_gain']
            
        except Exception as e:
            logging.error(f"计算负迁移指标时出错: {str(e)}")
            traceback.print_exc()
            metrics['is_negative_transfer'] = 'N/A'
            metrics['transfer_gain'] = 'N/A'
    
    return metrics


In [4]:
def evaluate_model(model, test_loader, model_name="Model", baseline_model=None, 
                   device='cuda', domain_idx=None, num_categories=None):
    """
    评估模型在测试集上的性能
    """
    from sklearn.metrics import mean_squared_error, r2_score
    
    # 自动检测num_categories
    if num_categories is None:
        for batch in test_loader:
            if len(batch) >= 4:
                category_onehot = batch[3]
                num_categories = category_onehot.shape[-1]
                break
        if num_categories is None:
            num_categories = 6  # 默认值
    
    def calculate_improved_mape(y_true, y_pred, epsilon=0.01):
        mask = np.abs(y_true) > epsilon
        if not np.any(mask):
            return float('nan')
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    model.eval()
    all_predictions = []
    all_targets = []
    all_lower_bounds = []
    all_upper_bounds = []
    all_uncertainties = []
    source_features = []
    target_features = []
    source_outputs = []
    target_outputs = []
    # ===== ✅ 基线模型变量初始化（在这里添加）=====
    baseline_source_features = []
    baseline_target_features = []
    baseline_source_outputs = []
    baseline_target_outputs = []
    all_baseline_lower_bounds = []
    all_baseline_upper_bounds = []
    all_baseline_uncertainties = []
    # ================================================
    
    with torch.no_grad():
        for batch_data in tqdm(test_loader, desc=f"Evaluating {model_name}"):
            # 解包数据
            if len(batch_data) == 4:
                inputs, targets, category_name, category_onehot = batch_data
            else:
                inputs, targets = batch_data[:2]
                category_onehot = batch_data[3] if len(batch_data) > 3 else None
            
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            
            if category_onehot is not None:
                category_onehot = category_onehot.float().to(device)
            else:
                batch_size = inputs.shape[0]
                category_onehot = torch.zeros(batch_size, num_categories).to(device)
                category_onehot[:, 0] = 1
            
            try:
                # ✅ 修复：提取域特征时不要reshape，直接使用原始输入
                if hasattr(model, 'time_series_encoder'):
                    try:
                        # 直接传递原始输入，让模型内部处理维度
                        # 不要在这里reshape！
                        source_feat = model.time_series_encoder(inputs, domain_idx=0)
                        
                        # 处理不同维度的特征输出
                        if len(source_feat.shape) == 4:
                            # [batch, buildings, seq, hidden] -> [batch*buildings, hidden]
                            source_feat = source_feat.mean(dim=2).reshape(-1, source_feat.shape[-1])
                        elif len(source_feat.shape) == 3:
                            # [batch, seq, hidden] -> [batch, hidden]
                            source_feat = source_feat.mean(dim=1)
                        
                        source_features.append(source_feat.cpu())
                        
                        target_feat = model.time_series_encoder(inputs, domain_idx=1)
                        if len(target_feat.shape) == 4:
                            target_feat = target_feat.mean(dim=2).reshape(-1, target_feat.shape[-1])
                        elif len(target_feat.shape) == 3:
                            target_feat = target_feat.mean(dim=1)
                        
                        target_features.append(target_feat.cpu())
                        
                    except Exception as e:
                        # 静默处理或使用更详细的错误信息
                        if "domain_idx" in str(e):
                            # 可能模型不支持domain_idx参数
                            pass
                        else:
                            print(f"提取域特征时出错: {str(e)}")
                
                # 不确定性预测
                mean_pred, lower_bound, upper_bound, uncertainty = predict_with_uncertainty(
                    model=model, 
                    inputs=inputs, 
                    category_onehot=category_onehot,
                    domain_idx=domain_idx, 
                    device=device, 
                    mc_samples=50
                )
                
                if isinstance(mean_pred, torch.Tensor):
                    mean_pred = mean_pred.cpu().numpy()
                
                # 收集域输出
                if hasattr(model, 'time_series_encoder'):
                    source_outputs.append(mean_pred)
                    try:
                        target_pred = model(inputs, category_onehot, domain_idx=domain_idx)
                        if isinstance(target_pred, torch.Tensor):
                            target_pred = target_pred.cpu().numpy()
                        target_outputs.append(target_pred)
                    except:
                        target_outputs.append(mean_pred)
                
                all_predictions.append(mean_pred)
                all_targets.append(targets.cpu().numpy())
                all_lower_bounds.append(lower_bound)
                all_upper_bounds.append(upper_bound)
                all_uncertainties.append(uncertainty)
                
            except Exception as e:
                print(f"批次处理错误: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
            
            # 新增：处理基线模型的特征和输出
            if baseline_model is not None:
                try:
                    if hasattr(baseline_model, 'time_series_encoder'):
                        try:
                            # 处理4D输入
                            if len(inputs.shape) == 4:
                                batch_size, num_buildings, seq_len, features = inputs.shape
                                inputs_reshaped = inputs.reshape(batch_size * num_buildings, seq_len, features)
                                baseline_source_feat = baseline_model.time_series_encoder(inputs_reshaped, domain_idx=0)
                                baseline_source_features.append(baseline_source_feat.mean(dim=1))
                                baseline_target_feat = baseline_model.time_series_encoder(inputs_reshaped, domain_idx=1)
                                baseline_target_features.append(baseline_target_feat.mean(dim=1))
                            else:
                                baseline_source_feat = baseline_model.time_series_encoder(inputs, domain_idx=0)
                                baseline_source_features.append(baseline_source_feat.mean(dim=1))
                                baseline_target_feat = baseline_model.time_series_encoder(inputs, domain_idx=1)
                                baseline_target_features.append(baseline_target_feat.mean(dim=1))
                        except Exception as e:
                            print(f"提取基线域特征时出错: {str(e)}")
                    
                    if hasattr(baseline_model, 'predict'):
                        baseline_pred = baseline_model.predict(inputs)
                    else:
                        baseline_pred = baseline_model(inputs, category_onehot)
                        if isinstance(baseline_pred, tuple):
                            baseline_pred = baseline_pred[0]
                    
                    if isinstance(baseline_pred, torch.Tensor):
                        baseline_pred = baseline_pred.cpu().numpy()
                    
                    baseline_source_outputs.append(baseline_pred)
                    baseline_target_outputs.append(baseline_pred)  # 基线模型没有域适应能力
                    
                    # 新增：处理基线模型不确定性
                    try:
                        if hasattr(baseline_model, 'predict_with_uncertainty'):
                            # 调用基线模型的不确定性预测方法
                            bl_mean, bl_lower, bl_upper, bl_uncertainty = predict_with_uncertainty(
                                model=baseline_model, 
                                inputs=inputs, 
                                category_onehot=category_onehot,  # 修改
                                domain_idx=domain_idx, 
                                device=device, 
                                mc_samples=50
                            )
                            all_baseline_lower_bounds.append(bl_lower)
                            all_baseline_upper_bounds.append(bl_upper)
                            all_baseline_uncertainties.append(bl_uncertainty)
                        else:
                            # 无不确定性方法时生成默认边界
                            bl_pred = baseline_pred.reshape(-1, 1)
                            all_baseline_lower_bounds.append(bl_pred * 0.9)
                            all_baseline_upper_bounds.append(bl_pred * 1.1)
                            all_baseline_uncertainties.append(np.ones_like(bl_pred) * 0.1)
                    except Exception as e:
                        print(f"基线不确定性计算错误: {e}")
                        # 生成默认不确定性
                        bl_pred = baseline_pred.reshape(-1, 1)
                        all_baseline_lower_bounds.append(bl_pred * 0.9)
                        all_baseline_upper_bounds.append(bl_pred * 1.1)
                        all_baseline_uncertainties.append(np.ones_like(bl_pred) * 0.1)
                        
                except Exception as e:
                    print(f"处理基线模型时出错: {str(e)}")
    
    if not all_predictions:
        print("警告: 无有效预测，无法评估")
        return {"Model": model_name, "Error": "无有效预测"}
    
    # 处理不同维度的数据
    try:
        # 检查并处理不同形状的数据
        if isinstance(all_predictions[0], np.ndarray) and len(all_predictions[0].shape) > 2:
            # 如果是3D或4D数据，需要展平
            all_predictions = [p.reshape(-1, p.shape[-1]) if len(p.shape) > 2 else p for p in all_predictions]
            all_targets = [t.reshape(-1, t.shape[-1]) if len(t.shape) > 2 else t for t in all_targets]
        
        all_predictions = np.concatenate(all_predictions, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)
    except Exception as e:
        print(f"合并数据错误: {str(e)}")
        print(f"预测形状示例: {all_predictions[0].shape if all_predictions else 'None'}")
        print(f"目标形状示例: {all_targets[0].shape if all_targets else 'None'}")
        return {"Model": model_name, "Error": f"合并失败: {str(e)}"}
    
    # 处理不确定性边界
    if not all_lower_bounds or not all_upper_bounds or not all_uncertainties:
        print("警告: 使用默认不确定性估计")
        all_lower_bounds = all_predictions * 0.9
        all_upper_bounds = all_predictions * 1.1
        all_uncertainties = np.ones_like(all_predictions) * 0.1
    else:
        try:
            # 处理不同形状的不确定性数据
            if isinstance(all_lower_bounds[0], np.ndarray) and len(all_lower_bounds[0].shape) > 2:
                all_lower_bounds = [b.reshape(-1, b.shape[-1]) if len(b.shape) > 2 else b for b in all_lower_bounds]
                all_upper_bounds = [b.reshape(-1, b.shape[-1]) if len(b.shape) > 2 else b for b in all_upper_bounds]
                all_uncertainties = [u.reshape(-1, u.shape[-1]) if len(u.shape) > 2 else u for u in all_uncertainties]
            
            all_lower_bounds = np.concatenate(all_lower_bounds, axis=0)
            all_upper_bounds = np.concatenate(all_upper_bounds, axis=0)
            all_uncertainties = np.concatenate(all_uncertainties, axis=0)
        except Exception as e:
            print(f"合并不确定性错误: {str(e)}")
            all_lower_bounds = all_predictions * 0.9
            all_upper_bounds = all_predictions * 1.1
            all_uncertainties = np.ones_like(all_predictions) * 0.1
    
    # 展平数据用于指标计算
    all_predictions_flat = all_predictions.reshape(-1)
    all_targets_flat = all_targets.reshape(-1)
    
    if all_lower_bounds.size != all_predictions_flat.size:
        print(f"形状不匹配: 下界{all_lower_bounds.shape} vs 预测{all_predictions.shape}")
        all_lower_bounds = all_lower_bounds.reshape(-1)[:all_predictions_flat.size]
        all_upper_bounds = all_upper_bounds.reshape(-1)[:all_predictions_flat.size]
        all_uncertainties = all_uncertainties.reshape(-1)[:all_predictions_flat.size]
    else:
        all_lower_bounds = all_lower_bounds.reshape(-1)
        all_upper_bounds = all_upper_bounds.reshape(-1)
        all_uncertainties = all_uncertainties.reshape(-1)
    
    print("\n数据形状:")
    print(f"预测: {all_predictions_flat.shape}, 目标: {all_targets_flat.shape}")
    print(f"下界: {all_lower_bounds.shape}, 上界: {all_upper_bounds.shape}")
    
    # 处理基线模型（保持原有逻辑，只修改数据解包部分）
    baseline_predictions = None
    baseline_metrics = {}
    if baseline_model is not None:
        baseline_model.eval()
        all_baseline_predictions = []
        with torch.no_grad():
            for batch_data in test_loader:
                try:
                    # 修改：解包新的数据格式
                    if len(batch_data) == 4:
                        inputs, _, category_name, category_onehot = batch_data
                    else:
                        inputs = batch_data[0]
                        category_onehot = batch_data[3] if len(batch_data) > 3 else None
                    
                    inputs = inputs.float().to(device)
                    if category_onehot is not None:
                        category_onehot = category_onehot.float().to(device)
                    else:
                        batch_size = inputs.shape[0]
                        category_onehot = torch.zeros(batch_size, num_categories).to(device)
                        category_onehot[:, 0] = 1
                    
                    if hasattr(baseline_model, 'predict'):
                        baseline_pred = baseline_model.predict(inputs)
                    else:
                        baseline_pred = baseline_model(inputs, category_onehot)
                        if isinstance(baseline_pred, tuple):
                            baseline_pred = baseline_pred[0]
                    if isinstance(baseline_pred, torch.Tensor):
                        baseline_pred = baseline_pred.cpu().numpy()
                    all_baseline_predictions.append(baseline_pred)
                except Exception as e:
                    print(f"基线预测错误: {str(e)}")
        if all_baseline_predictions:
            try:
                baseline_predictions = np.concatenate(all_baseline_predictions, axis=0).reshape(-1)
                if baseline_predictions.size != all_predictions_flat.size:
                    print(f"基线形状不匹配: {baseline_predictions.shape} vs {all_predictions_flat.shape}")
                    baseline_predictions = baseline_predictions[:all_predictions_flat.size]
                
                # 合并不确定性数据
                if all_baseline_lower_bounds:
                    try:
                        all_baseline_lower_bounds = np.concatenate(all_baseline_lower_bounds, axis=0).reshape(-1)
                        all_baseline_upper_bounds = np.concatenate(all_baseline_upper_bounds, axis=0).reshape(-1)
                        all_baseline_uncertainties = np.concatenate(all_baseline_uncertainties, axis=0).reshape(-1)
                        
                        # 确保形状匹配
                        if all_baseline_lower_bounds.size != baseline_predictions.size:
                            all_baseline_lower_bounds = all_baseline_lower_bounds[:baseline_predictions.size]
                            all_baseline_upper_bounds = all_baseline_upper_bounds[:baseline_predictions.size]
                            all_baseline_uncertainties = all_baseline_uncertainties[:baseline_predictions.size]
                    except Exception as e:
                        print(f"合并不确定性错误: {e}")
                        # fallback到默认值
                        all_baseline_lower_bounds = baseline_predictions * 0.9
                        all_baseline_upper_bounds = baseline_predictions * 1.1
                        all_baseline_uncertainties = np.ones_like(baseline_predictions) * 0.1
                else:
                    # 无收集数据时使用默认值
                    all_baseline_lower_bounds = baseline_predictions * 0.9
                    all_baseline_upper_bounds = baseline_predictions * 1.1
                    all_baseline_uncertainties = np.ones_like(baseline_predictions) * 0.1
                
                # 计算基线模型指标（包含不确定性）
                baseline_metrics = calculate_metrics(
                    predictions=baseline_predictions,
                    real_values=all_targets_flat,
                    model_name=f"Baseline for {model_name}",
                    baseline_predictions=None,  # 基线无需自身基线
                    lower_bounds=all_baseline_lower_bounds,
                    upper_bounds=all_baseline_upper_bounds,
                    confidence=0.95
                )
                # 补充基线模型的MAPE计算（使用改进版）
                baseline_metrics['MAPE'] = calculate_improved_mape(all_targets_flat, baseline_predictions, epsilon=0.01)
                baseline_metrics['avg_uncertainty'] = np.mean(all_baseline_uncertainties)
            except Exception as e:
                print(f"处理基线错误: {str(e)}")
                baseline_predictions = None
    
    try:
        metrics = calculate_metrics(
            predictions=all_predictions_flat,
            real_values=all_targets_flat,
            model_name=model_name,
            baseline_predictions=baseline_predictions,
            lower_bounds=all_lower_bounds,
            upper_bounds=all_upper_bounds,
            confidence=0.95
        )
        
        metrics['avg_uncertainty'] = np.mean(all_uncertainties)
        
        try:
            metrics['MAPE'] = calculate_improved_mape(all_targets_flat, all_predictions_flat, epsilon=0.01)
        except Exception as e:
            print(f"MAPE计算错误: {str(e)}")
            if 'MAPE' not in metrics:
                metrics['MAPE'] = float('nan')
    
        # 迁移学习指标计算
        if source_features and target_features and source_outputs and target_outputs:
            try:
                # 计算迁移模型的迁移指标
                source_features_tensor = torch.cat(source_features, dim=0).cpu()
                target_features_tensor = torch.cat(target_features, dim=0).cpu()
                source_outputs = np.concatenate(source_outputs, axis=0)
                target_outputs = np.concatenate(target_outputs, axis=0)
                
                transfer_metrics = calculate_transfer_metrics(
                    source_features=source_features_tensor,
                    target_features=target_features_tensor,
                    source_outputs=source_outputs,
                    target_outputs=target_outputs,
                    baseline_predictions=baseline_predictions,
                    targets=all_targets_flat
                )
                metrics.update(transfer_metrics)
                
                # 计算基线模型的迁移指标
                # ✅ 修复代码
                if baseline_source_features and baseline_target_features and baseline_source_outputs and baseline_target_outputs:
                    try:
                        baseline_source_features_tensor = torch.cat(baseline_source_features, dim=0).cpu()
                        baseline_target_features_tensor = torch.cat(baseline_target_features, dim=0).cpu()
                        
                        # ✅ 使用新变量名，不覆盖原始列表
                        baseline_source_outputs_array = np.concatenate(baseline_source_outputs, axis=0)
                        baseline_target_outputs_array = np.concatenate(baseline_target_outputs, axis=0)
                        
                        baseline_transfer_metrics = calculate_transfer_metrics(
                            source_features=baseline_source_features_tensor,
                            target_features=baseline_target_features_tensor,
                            source_outputs=baseline_source_outputs_array,  # ✅ 使用新变量
                            target_outputs=baseline_target_outputs_array,  # ✅ 使用新变量
                            baseline_predictions=baseline_predictions,
                            targets=all_targets_flat
                        )
                        
                        baseline_transfer_metrics = {f"baseline_{k}": v for k, v in baseline_transfer_metrics.items()}
                        metrics.update(baseline_transfer_metrics)
                        
                        # 计算改进率
                        if 'a_distance' in transfer_metrics and 'baseline_a_distance' in metrics:
                            metrics['a_distance_improvement'] = metrics['baseline_a_distance'] - metrics['a_distance']
                        if 'mmd' in transfer_metrics and 'baseline_mmd' in metrics:
                            metrics['mmd_improvement'] = metrics['baseline_mmd'] - metrics['mmd']
                        if 'feature_alignment' in transfer_metrics and 'baseline_feature_alignment' in metrics:
                            metrics['alignment_improvement'] = metrics['feature_alignment'] - metrics['baseline_feature_alignment']
                    
                    except Exception as e:
                        print(f"计算基线迁移指标时出错: {str(e)}")
                else:
                    print("警告: 基线模型的迁移学习数据缺失，跳过相关指标")
            except Exception as e:
                print(f"迁移指标错误: {str(e)}")
        else:
            print("警告: 迁移学习数据缺失，跳过相关指标")
    
    except Exception as e:
        print(f"指标计算错误: {str(e)}")
        rmse = np.sqrt(mean_squared_error(all_targets_flat, all_predictions_flat))
        r2 = r2_score(all_targets_flat, all_predictions_flat) if all_predictions_flat.size > 0 and all_targets_flat.size > 0 else np.nan
        metrics = {
            'Model': model_name,
            'RMSD': rmse,
            'R2': r2,
            'avg_uncertainty': np.mean(all_uncertainties)
        }
        try:
            metrics['MAPE'] = calculate_improved_mape(all_targets_flat, all_predictions_flat, epsilon=0.01)
        except:
            metrics['MAPE'] = float('nan')
    
    print(f"\n{model_name} 评估报告:")
    print(f"RMSD: {metrics.get('RMSD', 'N/A'):.4f}")
    print(f"MAPE: {metrics.get('MAPE', 'N/A'):.2f}%")
    print(f"R²: {metrics.get('R2', 'N/A'):.4f}")
    print(f"CV-RMSE: {metrics.get('CV-RMSE', 'N/A'):.4f}%")
    print(f"SD_real: {metrics.get('SD_real', 'N/A'):.4f}")
    print(f"SD_pred: {metrics.get('SD_pred', 'N/A'):.4f}")
    print(f"CC: {metrics.get('CC', 'N/A'):.4f}")
    
    if 'PIR' in metrics and metrics['PIR'] is not None:
        pir = metrics.get('PIR', 'N/A')
        if isinstance(pir, (int, float)):
            print(f"RMSD改进率(PIR): {pir:.2f}%")
    
    if baseline_metrics:
        print(f"\n{'Baseline Model' + ' ' * (len(model_name) - 11)} 评估报告:")
        print(f"RMSD: {baseline_metrics.get('RMSD', 'N/A'):.4f}")
        print(f"MAPE: {baseline_metrics.get('MAPE', 'N/A'):.2f}%")
        print(f"R²: {baseline_metrics.get('R2', 'N/A'):.4f}")
        print(f"CV-RMSE: {baseline_metrics.get('CV-RMSE', 'N/A'):.4f}%")
        print(f"SD_real: {baseline_metrics.get('SD_real', 'N/A'):.4f}")
        print(f"SD_pred: {baseline_metrics.get('SD_pred', 'N/A'):.4f}")
        print(f"CC: {baseline_metrics.get('CC', 'N/A'):.4f}")
        
        # 安全打印不确定性指标（避免字符串格式化错误）
        if 'PICP' in baseline_metrics:
            print("\n基线模型不确定性评估:")
            picp = baseline_metrics.get('PICP', 'N/A')
            print(f"PICP: {picp:.2f}%" if isinstance(picp, (int, float)) else f"PICP: {picp}")
            
            nmpiw = baseline_metrics.get('NMPIW', 'N/A')
            print(f"NMPIW: {nmpiw:.4f}" if isinstance(nmpiw, (int, float)) else f"NMPIW: {nmpiw}")
            
            cal_err = baseline_metrics.get('calibration_error', 'N/A')
            print(f"校准误差: {cal_err:.2f}%" if isinstance(cal_err, (int, float)) else f"校准误差: {cal_err}")
            
            uncertainty = baseline_metrics.get('avg_uncertainty', 'N/A')
            print(f"平均不确定性: {uncertainty:.4f}" if isinstance(uncertainty, (int, float)) else f"平均不确定性: {uncertainty}")
    
    # 打印迁移学习评估对比
    if any(key in metrics for key in ['a_distance', 'feature_alignment', 'mmd']):
        print("\n迁移学习评估:")
        print(f"A-distance: {metrics.get('a_distance', 'N/A'):.4f}")
        print(f"特征对齐: {metrics.get('feature_alignment', 'N/A'):.4f}")
        print(f"MMD: {metrics.get('mmd', 'N/A'):.4f}")
    
    if 'PICP' in metrics:
        print("\n不确定性评估:")
        picp = metrics.get('PICP', 'N/A')
        print(f"PICP: {picp:.2f}%" if isinstance(picp, (int, float)) else f"PICP: {picp}")
        
        nmpiw = metrics.get('NMPIW', 'N/A')
        print(f"NMPIW: {nmpiw:.4f}" if isinstance(nmpiw, (int, float)) else f"NMPIW: {nmpiw}")
        
        cal_err = metrics.get('calibration_error', 'N/A')
        print(f"校准误差: {cal_err:.2f}%" if isinstance(cal_err, (int, float)) else f"校准误差: {cal_err}")
        
        uqs = metrics.get('UQS', 'N/A')
        print(f"UQS: {uqs:.4f}" if isinstance(uqs, (int, float)) else f"UQS: {uqs}")
        
        uncertainty = metrics.get('avg_uncertainty', 'N/A')
        print(f"平均不确定性: {uncertainty:.4f}" if isinstance(uncertainty, (int, float)) else f"平均不确定性: {uncertainty}")
    
    try:
        sample_idx = 0
        if len(all_predictions.shape) == 3 and all_predictions.shape[0] > 0 and all_predictions.shape[1] > 0 and all_predictions.shape[2] > 0:
            max_horizon = min(10, all_predictions.shape[2])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_horizon),
                    all_lower_bounds.reshape(all_predictions.shape)[sample_idx, 0, :max_horizon],
                    all_upper_bounds.reshape(all_predictions.shape)[sample_idx, 0, :max_horizon],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_horizon), all_predictions[sample_idx, 0, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, 0, :max_horizon], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"3D data visualization error: {str(e)}")
                plt.plot(range(max_horizon), all_predictions[sample_idx, 0, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, 0, :max_horizon], 'r-x', label='Ground Truth')
        
        elif len(all_predictions.shape) == 2 and all_predictions.shape[0] > 0 and all_predictions.shape[1] > 0:
            max_horizon = min(10, all_predictions.shape[1])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_horizon),
                    all_lower_bounds.reshape(all_predictions.shape)[sample_idx, :max_horizon],
                    all_upper_bounds.reshape(all_predictions.shape)[sample_idx, :max_horizon],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_horizon), all_predictions[sample_idx, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, :max_horizon], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"2D data visualization error: {str(e)}")
                plt.plot(range(max_horizon), all_predictions[sample_idx, :max_horizon], 'b-o', label='Predictions')
                plt.plot(range(max_horizon), all_targets[sample_idx, :max_horizon], 'r-x', label='Ground Truth')
        
        elif len(all_predictions.shape) == 1 and all_predictions.shape[0] > 0:
            max_points = min(10, all_predictions.shape[0])
            plt.figure(figsize=(10, 6))
            try:
                plt.fill_between(
                    range(max_points),
                    all_lower_bounds[:max_points],
                    all_upper_bounds[:max_points],
                    alpha=0.3, color='blue', label='95% Confidence Interval'
                )
                plt.plot(range(max_points), all_predictions[:max_points], 'b-o', label='Predictions')
                plt.plot(range(max_points), all_targets[:max_points], 'r-x', label='Ground Truth')
            except Exception as e:
                print(f"1D data visualization error: {str(e)}")
                plt.plot(range(max_points), all_predictions[:max_points], 'b-o', label='Predictions')
                plt.plot(range(max_points), all_targets[:max_points], 'r-x', label='Ground Truth')
        
        else:
            print(f"Cannot visualize predictions: incompatible shape {all_predictions.shape}")
            return metrics
        
        plt.title(f'{model_name} Prediction Example\nPICP: {metrics.get("PICP", "N/A")}%')
        plt.xlabel('Prediction Time Step')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        safe_model_name = ''.join(c if c.isalnum() or c in ['-', '_'] else '_' for c in model_name)
        plt.savefig(f'prediction_sample_{safe_model_name}.png')
        plt.close()
    except Exception as e:
        print(f"可视化错误: {str(e)}")
        print(f"数据形状: 预测={all_predictions.shape}, 目标={all_targets.shape}")
    
    if baseline_metrics:
        metrics['baseline_metrics'] = baseline_metrics
    
    return metrics

## 使用全部类别进行源域模型训练

In [5]:
from bilstm import BaselineBiLSTM
from AdaptiveBILSTM import AdaptiveBiLSTM  # 确保导入
import os
import json
import torch
import torch.nn as nn  # 添加nn导入
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import logging
from tqdm.auto import tqdm
import warnings
import copy
from torch import optim
from torch.utils.data import Subset, DataLoader, random_split

# 修改：导入新的数据加载函数
from dataloader import (
    load_source_domain_dataloaders,
    load_transfer_learning_dataloaders
)

# 配置日志：同时输出到文件和控制台，记录时间、等级和消息
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelna me)s - %(message)s',
    handlers=[
        logging.FileHandler('training.log'),  # 日志文件
        logging.StreamHandler()               # 控制台输出
    ]
)

# 忽略警告
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# ============ 模型参数设置 ============
batch_size = 32
sequence_length = 24
forecast_horizon = 24
hidden_dim = 64
num_layers = 2
dropout = 0.2
learning_rate = 0.001
weight_decay = 1e-4
epochs = 10
input_dim = 6  # 1个电力特征 + 5个天气特征

# 定义Huber损失函数
class HuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta
    
    def forward(self, predictions, targets):
        # 计算预测值与目标值之间的差异
        diff = predictions - targets
        abs_diff = torch.abs(diff)
        
        # 应用Huber损失公式
        loss = torch.where(
            abs_diff <= self.delta,
            0.5 * diff * diff,
            self.delta * (abs_diff - 0.5 * self.delta)
        )
        
        return torch.mean(loss)

# 计算损失的辅助函数
def calculate_loss(predictions, targets):
    """
    计算Huber损失
    
    参数:
    - predictions: 预测值
    - targets: 真实值
    
    返回:
    - loss: Huber损失值
    """
    loss_fn = HuberLoss(delta=1.0)
    return loss_fn(predictions, targets)

def train_and_save_model(model, train_loader, val_loader, epochs, lr, weight_decay, 
                       model_name, save_dir='models', device='cuda', early_stopping_patience=5,
                       source_domain_idx=0, target_domain_idx=None):
    """
    训练模型并保存最佳模型，支持域自适应
    
    新增参数:
    - source_domain_idx: 源域索引，默认为0
    - target_domain_idx: 用于验证，默认为0，使用源域验证
    """
    os.makedirs(save_dir, exist_ok=True)
    model = model.to(device)
    
    # 检查train_loader是否为None（CC类别的情况）
    if train_loader is None:
        print(f"⚠️ 警告：训练数据加载器为空（可能是CC类别），跳过训练")
        return model, None, None
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    # 初始化记录器
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    
    # 保存模型配置信息（不同模型有不同的属性）
    model_config = {}
    
    # 尝试获取常见模型属性（如果存在的话）
    for attr in ['input_dim', 'hidden_dim', 'forecast_horizon', 'num_buildings', 'num_layers', 'dropout']:
        if hasattr(model, attr):
            model_config[attr] = getattr(model, attr)
    
    # 根据模型类型获取特定属性
    is_adaptive_model = hasattr(model, 'time_series_encoder')
    if is_adaptive_model:
        model_config['model_type'] = 'AdaptiveBiLSTM'
        if hasattr(model.time_series_encoder, 'num_domains'):
            model_config['num_domains'] = model.time_series_encoder.num_domains
        model_config['source_domain_idx'] = source_domain_idx
        model_config['target_domain_idx'] = target_domain_idx
    
    # 保存最佳模型的信息
    best_model_info = {
        'state_dict': None,
        'optimizer_state': None,
        'epoch': 0,
        'train_loss': float('inf'),
        'val_loss': float('inf'),
        'metrics': None
    }

    print(f"开始训练 {model_name}...")

    for epoch in range(epochs):
        # 训练阶段
        model.train()
        epoch_train_loss = 0
        train_steps = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Training]', mininterval=3.0)
        for batch in progress_bar:
            # 修改：解包新的数据格式 (features, targets, category_name, category_onehot)
            inputs, targets, category_name, category_onehot = batch
            
            # 将数据移动到设备
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            category_onehot = category_onehot.float().to(device)
            
            optimizer.zero_grad()
            
            # 根据模型类型调用
            if is_adaptive_model:
                predictions = model(inputs, category_onehot, domain_idx=source_domain_idx)
            else:
                predictions = model(inputs, category_onehot)
                
            loss = calculate_loss(predictions, targets)  # 使用Huber损失函数
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_train_loss += loss.item()
            train_steps += 1
            progress_bar.set_postfix({'train_loss': f'{loss.item():.4f}'})
        
        avg_train_loss = epoch_train_loss / train_steps
        train_losses.append(avg_train_loss)
        
        # 验证阶段
        if val_loader is not None:
            model.eval()
            epoch_val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                progress_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Validation]')
                for batch in progress_bar:
                    # 修改：解包新的数据格式
                    inputs, targets, category_name, category_onehot = batch
                    
                    inputs = inputs.float().to(device)
                    targets = targets.float().to(device)
                    category_onehot = category_onehot.float().to(device)
                    
                    if is_adaptive_model:
                        predictions = model(inputs, category_onehot, domain_idx=source_domain_idx)
                    else:
                        predictions = model(inputs, category_onehot)
                    
                    loss = calculate_loss(predictions, targets)
                    
                    epoch_val_loss += loss.item()
                    val_steps += 1
                    progress_bar.set_postfix({'val_loss': f'{loss.item():.4f}'})
            
            avg_val_loss = epoch_val_loss / val_steps
            val_losses.append(avg_val_loss)
            
            # 学习率调整
            scheduler.step(avg_val_loss)
            
            # 计算当前模型的评估指标
            current_metrics = simple_evaluate_model(
                model, val_loader, f"{model_name}_epoch_{epoch+1}", 
                device=device, domain_idx=target_domain_idx if is_adaptive_model else None
            )
            
            # 检查是否是最佳模型
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_epoch = epoch
                patience_counter = 0
                
                # 更新最佳模型信息
                best_model_info = {
                    'state_dict': copy.deepcopy(model.state_dict()),
                    'optimizer_state': copy.deepcopy(optimizer.state_dict()),
                    'epoch': epoch + 1,
                    'train_loss': avg_train_loss,
                    'val_loss': avg_val_loss,
                    'metrics': current_metrics,
                    'hyperparameters': {
                        'lr': lr,
                        'weight_decay': weight_decay,
                        'epochs': epochs,
                        'best_epoch': epoch + 1,
                        'model_config': model_config,
                        'model_type': type(model).__name__,
                        'source_domain_idx': source_domain_idx if is_adaptive_model else None,
                        'target_domain_idx': target_domain_idx if is_adaptive_model else None
                    }
                }
                
                # 保存最佳模型检查点
                checkpoint_path = os.path.join(save_dir, f'{model_name}_best.pth')
                torch.save(best_model_info, checkpoint_path)
                print(f"✅ 保存最佳模型 (epoch {epoch+1}), 验证损失: {avg_val_loss:.4f}")
            else:
                patience_counter += 1
            
            # 打印当前epoch的训练信息
            print(
                f"Epoch {epoch+1}/{epochs} - "
                f"Train Loss: {avg_train_loss:.4f}, "
                f"Val Loss: {avg_val_loss:.4f}, "
                f"RMSD: {current_metrics['RMSD']:.4f}, "
                f"R²: {current_metrics['R2']:.4f}, "
                f"Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch+1}), "
                f"LR: {optimizer.param_groups[0]['lr']:.6f}"
            )
            
            # 早停检查
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping triggered after epoch {epoch+1}")
                break
        else:
            # 没有验证集时只打印训练损失
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}")
    
    # 训练结束后，保存训练历史
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_epoch': best_epoch + 1,
        'best_val_loss': best_val_loss
    }
    
    # 保存训练历史
    history_path = os.path.join(save_dir, f'{model_name}_training_history.json')
    with open(history_path, 'w') as f:
        serializable_history = {
            'train_losses': [float(loss) for loss in train_losses],
            'val_losses': [float(loss) for loss in val_losses],
            'best_epoch': best_epoch + 1,
            'best_val_loss': float(best_val_loss)
        }
        json.dump(serializable_history, f, indent=4)
    
    # 绘制训练曲线（如果有验证集）
    if val_losses:
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Model (Epoch {best_epoch+1})')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} Training History')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(save_dir, f'{model_name}_training_curve.png'))
        plt.close()
    
    # 恢复最佳模型状态
    if best_model_info['state_dict'] is not None:
        model.load_state_dict(best_model_info['state_dict'])
    
    return model, best_model_info, history

# 修改简化评估函数
def simple_evaluate_model(model, test_loader, model_name="Model", device='cuda', domain_idx=None):
    """
    训练中使用的简化评估函数，支持域自适应
    """
    from sklearn.metrics import mean_squared_error, r2_score
    
    if test_loader is None:
        print("⚠️ 测试数据加载器为空")
        return {
            "Model": model_name,
            "RMSD": float('inf'),
            "R2": 0.0,
            "MAPE": float('inf'),
            "CC": 0.0
        }
    
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            # 修改：解包新的数据格式
            inputs, targets, category_name, category_onehot = batch
            
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            category_onehot = category_onehot.float().to(device)
            
            # 根据是否是自适应模型选择调用方式
            if hasattr(model, 'time_series_encoder'):
                predictions = model(inputs, category_onehot, domain_idx=domain_idx)
            else:
                predictions = model(inputs, category_onehot)
            
            # 收集预测和目标值
            pred_values = predictions.cpu().numpy()
            target_values = targets.cpu().numpy()
            
            all_preds.append(pred_values)
            all_targets.append(target_values)
    
    # 合并所有批次的预测和目标
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    # 展平数组以便计算指标
    all_preds = all_preds.reshape(-1)
    all_targets = all_targets.reshape(-1)
    
    # 计算基本指标
    mse = mean_squared_error(all_targets, all_preds)
    rmsd = np.sqrt(mse)
    r2 = r2_score(all_targets, all_preds)
    
    # 计算MAPE（排除零值）
    mask = np.abs(all_targets) > 1e-6
    mape = np.mean(np.abs((all_targets[mask] - all_preds[mask]) / all_targets[mask])) * 100 if np.any(mask) else np.nan
    
    # 计算CC (相关系数)
    cc = np.corrcoef(all_preds, all_targets)[0, 1]
    
    return {
        "Model": model_name,
        "RMSD": rmsd,
        "R2": r2,
        "MAPE": mape,
        "CC": cc
    }

# ============ 主训练流程 ============

# 创建保存目录
os.makedirs('models', exist_ok=True)

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 修改：使用新的数据加载方式
print("\n" + "="*70)
print("加载训练数据（源域）...")
print("="*70)

# 选择数据稀缺程度
data_shortage = 'mild'  # 可以选择 'mild', 'heavy', 'extreme'

# 加载所有训练建筑的数据（使用完整数据）
train_loader, val_loader, categories = load_source_domain_dataloaders(
    batch_size=batch_size,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    handle_missing='forward_fill',
    val_ratio=0.2  # 20% 用于验证
)

# 检查数据加载器
if train_loader is None:
    print("❌ 错误：无法加载训练数据")
    exit(1)

# 从train_loader中划分训练集和验证集
print("\n划分训练集和验证集...")
all_train_dataset = train_loader.dataset

# 设定划分比例（如80%训练，20%验证）
train_ratio = 0.8
val_ratio = 0.2
total_len = len(all_train_dataset)
train_len = int(total_len * train_ratio)
val_len = total_len - train_len

print(f"总样本数: {total_len}")
print(f"训练样本数: {train_len} ({train_ratio*100:.0f}%)")
print(f"验证样本数: {val_len} ({val_ratio*100:.0f}%)")

# 使用random_split划分
train_subset, val_subset = random_split(
    all_train_dataset, 
    [train_len, val_len],
    generator=torch.Generator().manual_seed(42)  # 设置种子以确保可重复性
)

# 构建新的dataloader
train_loader = DataLoader(
    train_subset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=True
)

val_loader = DataLoader(
    val_subset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=True
)

# 获取数据维度信息
print("\n检查数据维度...")
for batch in train_loader:
    inputs, targets, category_name, category_onehot = batch
    print(f"输入形状: {inputs.shape}")
    print(f"目标形状: {targets.shape}")
    print(f"类别One-hot形状: {category_onehot.shape}")
    
    # 提取维度信息
    if len(inputs.shape) == 4:  # [batch, buildings, seq_len, features]
        batch_size_actual = inputs.shape[0]
        num_buildings = inputs.shape[1]
        seq_len = inputs.shape[2]
        input_dim = inputs.shape[3]
    else:  # [batch, seq_len, features]
        batch_size_actual = inputs.shape[0]
        num_buildings = 1
        seq_len = inputs.shape[1]
        input_dim = inputs.shape[2]
    
    category_dim = category_onehot.shape[-1]
    
    print(f"批次大小: {batch_size_actual}")
    print(f"建筑数量: {num_buildings}")
    print(f"序列长度: {seq_len}")
    print(f"输入特征维度: {input_dim}")
    print(f"类别数量: {category_dim}")
    break

# 创建域自适应模型
print("\n" + "="*70)
print("创建AdaptiveBiLSTM模型...")
print("="*70)

num_domains = len(categories)  # 使用类别数作为域数
source_domain_idx = 0  # 源域索引
target_domain_idx = 0  # 评估时使用的仍然是源域

# 创建AdaptiveBiLSTM模型
source_model = AdaptiveBiLSTM(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    category_dim=category_dim,
    forecast_horizon=forecast_horizon,
    num_buildings=num_buildings,
    num_domains=num_domains,
    num_layers=num_layers,
    dropout=dropout
)

print(f"模型参数:")
print(f"  输入维度: {input_dim}")
print(f"  隐藏维度: {hidden_dim}")
print(f"  类别维度: {category_dim}")
print(f"  预测步长: {forecast_horizon}")
print(f"  建筑数量: {num_buildings}")
print(f"  域数量: {num_domains}")
print(f"  层数: {num_layers}")
print(f"  Dropout: {dropout}")

# 训练源域模型
print("\n" + "="*70)
print("开始训练源域模型...")
print("="*70)

trained_source_model, source_best_info, source_history = train_and_save_model(
    model=source_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=epochs,
    lr=learning_rate,
    weight_decay=weight_decay,
    model_name=f'adaptive_source_huber_{data_shortage}',
    save_dir='models',
    device=device,
    source_domain_idx=source_domain_idx,
    target_domain_idx=target_domain_idx
)


# 使用改进的完整评估函数对最终模型进行评估
print("\n" + "="*70)
print("评估源域模型性能...")
print("="*70)

source_metrics = evaluate_model(
    model=trained_source_model, 
    test_loader=val_loader,
    model_name=f"Adaptive_Source_Huber_Model_{data_shortage}",
    device=device,
    domain_idx=source_domain_idx
)

# 提取重要的评估指标
print("\n源域模型评估指标:")
print(f"MAPE(%): {source_metrics.get('MAPE', 'N/A'):.2f}")
print(f"RMSD: {source_metrics.get('RMSD', 'N/A'):.4f}")
print(f"R²: {source_metrics.get('R2', 'N/A'):.4f}")

def convert_to_serializable(obj):
    """将对象转换为JSON可序列化的格式"""
    if isinstance(obj, torch.Tensor):
        obj = obj.detach().cpu().numpy()
        if obj.size == 1:
            return float(obj.item())
        return obj.tolist()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64, np.int32, np.int64)):
        return float(obj) if np.issubdtype(obj.dtype, np.floating) else int(obj)
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (float, int, str, bool, type(None))):
        return obj
    else:
        try:
            return str(obj)
        except:
            return "Non-serializable object"

# 打印源域的不确定性评估结果
if 'PICP' in source_metrics:
    print("\n源域不确定性评估:")
    print(f"预测区间覆盖率(PICP): {source_metrics['PICP']:.2f}% (目标95%)")
    print(f"校准误差: {source_metrics.get('calibration_error', 'N/A'):.2f}%")
    print(f"平均区间宽度(NMPIW): {source_metrics.get('NMPIW', 'N/A'):.4f}")
    print(f"不确定性质量分数(UQS): {source_metrics.get('UQS', 'N/A'):.4f}")

# 保存最终评估结果
metrics_path = os.path.join('models', f'adaptive_model_evaluation_metrics_{data_shortage}.json')
with open(metrics_path, 'w') as f:
    all_metrics = {
        'source_domain': {k: convert_to_serializable(v) 
                         for k, v in source_metrics.items() if k != 'Model'},
        'model_info': convert_to_serializable({
            'best_epoch': source_best_info['epoch'] if source_best_info else 'N/A',
            'num_domains': num_domains,
            'model_type': 'AdaptiveBiLSTM',
            'data_shortage': data_shortage
        })
    }
    
    json.dump(all_metrics, f, indent=4)

print("\n" + "="*70)
print("✅ 训练完成！")
print("="*70)

使用设备: cuda

加载训练数据（源域）...
📊 加载源域训练数据
说明:
  • 仅使用train_test_labels.json中标记为train的建筑
  • 使用完整数据（无人为引入的缺失）
  • 按时间划分: 前80%用于训练, 后20%用于验证

🏢 建筑分组:
  训练建筑: 25 个

⚡ 加载电力数据...
  ✓ 完整电力数据: (17544, 25)
  ✓ 训练集电力: (17544, 25)

🌤️  加载天气数据...
  训练集天气数据:
    ✓ Hog: (17542, 5)
    ✓ Wolf: (17505, 5)
    ✓ Robin: (17516, 5)
    ✓ Eagle: (17536, 5)
    ✓ Rat: (17539, 5)

🔄 创建数据加载器...
  数据集: 25 个建筑, 13938 个有效样本 (first_80_percent)
  数据集: 25 个建筑, 3449 个有效样本 (last_20_percent)

✅ 源域数据加载完成!
训练样本: 13938 (前80%时间)
验证样本: 3449 (后20%时间)
批次大小: 32
缺失处理: forward_fill


划分训练集和验证集...
总样本数: 13938
训练样本数: 11150 (80%)
验证样本数: 2788 (20%)

检查数据维度...
输入形状: torch.Size([32, 25, 24, 6])
目标形状: torch.Size([32, 25, 24])
类别One-hot形状: torch.Size([32, 6])
批次大小: 32
建筑数量: 25
序列长度: 24
输入特征维度: 6
类别数量: 6

创建AdaptiveBiLSTM模型...
模型参数:
  输入维度: 6
  隐藏维度: 64
  类别维度: 6
  预测步长: 24
  建筑数量: 25
  域数量: 6
  层数: 2
  Dropout: 0.2

开始训练源域模型...
开始训练 adaptive_source_huber_mild...


Epoch 1/10 [Training]:   0%|          | 0/349 [00:00<?, ?it/s]

Epoch 1/10 [Validation]:   0%|          | 0/88 [00:00<?, ?it/s]

✅ 保存最佳模型 (epoch 1), 验证损失: 0.0082
Epoch 1/10 - Train Loss: 0.0127, Val Loss: 0.0082, RMSD: 0.1281, R²: 0.5511, Best Val Loss: 0.0082 (Epoch 1), LR: 0.001000


Epoch 2/10 [Training]:   0%|          | 0/349 [00:00<?, ?it/s]

...

Epoch 10/10 [Validation]:   0%|          | 0/88 [00:00<?, ?it/s]

Epoch 10/10 - Train Loss: 0.0046, Val Loss: 0.0045, RMSD: 0.0942, R²: 0.7575, Best Val Loss: 0.0042 (Epoch 9), LR: 0.001000

评估源域模型性能...


Evaluating Adaptive_Source_Huber_Model_mild:   0%|          | 0/88 [00:00<?, ?it/s]

INFO:root:采样后: source_features.shape=torch.Size([5000, 64]), target_features.shape=torch.Size([5000, 64])



数据形状:
预测: (1672800,), 目标: (1672800,)
下界: (1672800,), 上界: (1672800,)
警告: 基线模型的迁移学习数据缺失，跳过相关指标

Adaptive_Source_Huber_Model_mild 评估报告:
RMSD: 0.0914
MAPE: 15.89%
R²: 0.7717
CV-RMSE: 19.5614%
SD_real: 0.1912
SD_pred: 0.1588
CC: 0.8799

迁移学习评估:
A-distance: 0.9000
特征对齐: 0.6820
MMD: 0.5192

不确定性评估:
PICP: 60.64%
NMPIW: 0.11225299537181854
校准误差: 34.36%
UQS: 0.4307
平均不确定性: 0.028688104823231697

源域模型评估指标:
MAPE(%): 15.89
RMSD: 0.0914
R²: 0.7717

源域不确定性评估:
预测区间覆盖率(PICP): 60.64% (目标95%)
校准误差: 34.36%
平均区间宽度(NMPIW): 0.1123
不确定性质量分数(UQS): 0.4307

✅ 训练完成！


In [6]:
import traceback
import logging
import numpy as np
from tqdm.auto import tqdm
import warnings
import torch.nn.functional as F
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.autograd import Function
from datetime import datetime

from AdaptiveBILSTM import GradientReversalFunction,grad_reverse,DomainDiscriminator

# Huber损失 - 对异常值更鲁棒的损失函数
class HuberLoss(nn.Module):
    """
    Huber损失 - 对异常值更鲁棒的损失函数
    
    Args:
        delta: 阈值参数，控制MSE和MAE之间的平滑过渡
    """
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta
    
    def forward(self, predictions, targets):
        # 计算预测值与目标值之间的差异
        diff = predictions - targets
        abs_diff = torch.abs(diff)
        
        # 应用Huber损失公式
        loss = torch.where(
            abs_diff <= self.delta,
            0.5 * diff * diff,
            self.delta * (abs_diff - 0.5 * self.delta)
        )
        
        return torch.mean(loss)

def domain_adversarial_training_step(
    source_features, 
    target_features, 
    domain_discriminator, 
    optimizer_disc, 
    epoch, 
    total_epochs,
    device,
    projection_layer=None,
    target_domain_idx=1  # 目标域索引（仅用于数据标识，不影响二分类标签）
):
    """
    执行单个域对抗训练步骤（二分类逻辑，源域vs目标域）
    
    Args:
        source_features: 源域特征 [batch, buildings, seq_len, features] 或 [batch, seq_len, features]
        target_features: 目标域特征 [batch, buildings, seq_len, features] 或 [batch, seq_len, features]
        domain_discriminator: 域判别器模型
        optimizer_disc: 域判别器优化器
        epoch: 当前训练轮次
        total_epochs: 总训练轮次
        device: 计算设备
        projection_layer: 特征投影层（用于维度匹配）
        target_domain_idx: 目标域的索引值（仅用于数据层面的标识，标签统一为1）
    """
    try:
        # 打印输入特征形状用于调试
        logging.debug(f"Source features shape: {source_features.shape}")
        logging.debug(f"Target features shape: {target_features.shape}")
        
        # 处理4D特征格式 [batch, buildings, seq_len, features]
        if len(source_features.shape) == 4:
            batch_size, num_buildings, seq_len, feature_dim = source_features.shape
            # 将buildings维度合并到batch中
            source_features = source_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
            target_features = target_features.reshape(batch_size * num_buildings, seq_len, feature_dim)
            batch_size = source_features.size(0)  # 更新批次大小
        else:
            batch_size = source_features.size(0)
        
        # 确保特征维度匹配
        if source_features.size(-1) != target_features.size(-1):
            raise ValueError(f"特征维度不匹配: 源域={source_features.size(-1)}, 目标域={target_features.size(-1)}")
        
        # 分离特征以避免重复反向传播
        source_features = source_features.detach()
        target_features = target_features.detach()
        
        # 准备二分类标签（源域=0，目标域=1）
        source_domain_labels = torch.zeros(batch_size, 1).to(device)
        target_domain_labels = torch.ones(batch_size, 1).to(device)
        
        # 特征降维处理 - 在时间维度上取平均
        if len(source_features.shape) == 3:  # [batch, seq_len, hidden_dim]
            source_processed = source_features.mean(dim=1)  # [batch, hidden_dim]
            target_processed = target_features.mean(dim=1)  # [batch, hidden_dim]
        else:
            # 如果还有其他维度，先展平再处理
            source_processed = source_features.reshape(batch_size, -1)
            target_processed = target_features.reshape(batch_size, -1)
        
        # 维度匹配处理
        expected_dim = domain_discriminator.feature_dim
        if source_processed.size(-1) != expected_dim:
            if projection_layer is not None:
                source_processed = projection_layer(source_processed)
                target_processed = projection_layer(target_processed)
            else:
                # 动态创建投影层（如果未提供）
                if not hasattr(domain_adversarial_training_step, 'projection_layer'):
                    domain_adversarial_training_step.projection_layer = nn.Linear(
                        source_processed.size(-1), expected_dim
                    ).to(device)
                source_processed = domain_adversarial_training_step.projection_layer(source_processed)
                target_processed = domain_adversarial_training_step.projection_layer(target_processed)
        
        # 连接特征和标签
        features = torch.cat([source_processed, target_processed], dim=0)
        domain_labels = torch.cat([source_domain_labels, target_domain_labels], dim=0)
        
        # 梯度反转参数 - 随训练进度增加
        grad_reverse_strength = 2. / (1. + np.exp(-10 * epoch / total_epochs)) - 1
        reversed_features = grad_reverse(features, grad_reverse_strength)
        
        # 域判别器预测
        domain_preds = domain_discriminator.simple_model(reversed_features)
        
        # 计算二分类损失
        domain_loss = F.binary_cross_entropy_with_logits(domain_preds, domain_labels)
        
        # 更新判别器
        optimizer_disc.zero_grad()
        domain_loss.backward(retain_graph=True)
        optimizer_disc.step()
        
        # 分离预测结果
        source_domain_preds = domain_preds[:batch_size]
        target_domain_preds = domain_preds[batch_size:]
        
        return domain_loss, source_domain_preds, target_domain_preds
        
    except Exception as e:
        logging.warning(f"域对抗训练步骤出错: {str(e)}")
        traceback.print_exc()
        return (
            torch.tensor(0.0, device=device),
            torch.zeros(batch_size, 1, device=device),
            torch.zeros(batch_size, 1, device=device)
        )

# 自适应λ调度器 - 动态调整域对抗训练强度
def adaptive_lambda_scheduler(epoch, epochs, source_loss, target_loss, domain_loss, lambda_domain):
    """
    基于训练进度、任务损失和域判别损失动态调整梯度反转参数λ
    
    Args:
        epoch: 当前训练轮次
        epochs: 总训练轮次
        source_loss: 源域任务损失
        target_loss: 目标域任务损失
        domain_loss: 域判别损失
        lambda_domain: 基础λ值
    
    Returns:
        float: 调整后的λ值
    """
    # 1. 基于训练进度的基础调整
    progress = epoch / epochs
    
    # 训练初期：较小的λ值，专注于任务学习
    if progress < 0.3:
        base_lambda = max(0.001, lambda_domain * 0.01)
    # 训练中期：中等λ值，平衡任务学习和域适应
    elif progress < 0.7:
        base_lambda = max(0.005, lambda_domain * 0.05)
    # 训练后期：较大λ值，加强域适应
    else:
        base_lambda = max(0.01, lambda_domain * 0.1)
    
    # 2. 基于源域和目标域任务损失比例的调整
    task_ratio = target_loss / (source_loss + 1e-10)
    
    # 如果目标域损失远大于源域，减小λ以专注于任务学习
    if task_ratio > 2.0:
        adjust_factor = 0.5
    # 如果目标域损失远小于源域，增大λ以加强域适应
    elif task_ratio < 0.5:
        adjust_factor = 2.0
    else:
        adjust_factor = 1.0
    
    # 3. 基于域判别器性能的调整
    # 如果判别器损失接近0.693(log(2))，说明判别器无法区分域，减小λ
    if abs(domain_loss - 0.693) < 0.1:
        domain_factor = 0.8
    # 如果判别器损失太小，说明判别器过强，增大λ
    elif domain_loss < 0.3:
        domain_factor = 1.5
    else:
        domain_factor = 1.0
    
    # 计算最终λ值并限制在合理范围内
    final_lambda = base_lambda * adjust_factor * domain_factor
    return min(max(final_lambda, 0.001), 0.1)  # 限制范围 [0.001, 0.1]

import traceback as tb  # ✅ 在文件顶部导入，使用别名避免冲突

def adapt_to_target_domain(source_model, source_loader, target_loader, epochs=10, lr=0.001, 
                          device='cuda', lambda_domain=0.4, early_stopping_patience=5,
                          source_domain_idx=0, target_domain_idx=1):
    """
    修复版：域自适应迁移学习，处理不同形状的数据
    """
    
    if source_loader is None or target_loader is None:
        logging.error("数据加载器为空")
        return None, None
    
    os.makedirs('models', exist_ok=True)
    model = copy.deepcopy(source_model).to(device)
    is_adaptive_model = hasattr(model, 'time_series_encoder')
    
    # ✅ 修复：安全获取特征维度
    feature_dim = 64  # 默认值
    try:
        # 尝试从目标域获取样本（通常更简单）
        for batch in target_loader:
            sample_inputs = batch[0].float().to(device)
            sample_category = batch[3].float().to(device) if len(batch) > 3 else None
            
            with torch.no_grad():
                if is_adaptive_model:
                    sample_features = model.time_series_encoder(sample_inputs, domain_idx=target_domain_idx)
                else:
                    sample_features = model(sample_inputs, sample_category)
            
            feature_dim = sample_features.shape[-1]
            logging.info(f"从目标域确定特征维度: {feature_dim}")
            break
    except Exception as e:
        logging.warning(f"特征维度确定失败，使用默认值 {feature_dim}: {e}")
    
    # 初始化组件
    domain_discriminator = DomainDiscriminator(feature_dim=feature_dim, hidden_dim=64, dropout=0.3).to(device)
    projection_layer = nn.Linear(feature_dim, feature_dim).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    optimizer_disc = optim.AdamW(domain_discriminator.parameters(), lr=lr*0.5, weight_decay=0.02)
    optimizer_proj = optim.Adam(projection_layer.parameters(), lr=lr)
    
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=lr*0.1)
    
    huber_loss_fn = HuberLoss(delta=1.0)
    
    best_rmse = float('inf')
    patience_counter = 0
    best_model_state = None
    history = {'total_loss': [], 'task_loss': [], 'domain_loss': [], 'rmse': []}
    
    epoch_source_losses, epoch_target_losses, epoch_domain_losses = [], [], []
    
    logging.info("开始域自适应迁移学习...")
    
    for epoch in range(epochs):
        model.train()
        domain_discriminator.train()
        
        # 动态λ
        if epoch > 0 and epoch_source_losses:
            current_lambda = adaptive_lambda_scheduler(
                epoch-1, epochs, 
                sum(epoch_source_losses)/len(epoch_source_losses),
                sum(epoch_target_losses)/len(epoch_target_losses),
                sum(epoch_domain_losses)/len(epoch_domain_losses),
                lambda_domain
            )
        else:
            current_lambda = max(0.001, lambda_domain * 0.01)
        
        epoch_stats = {'total_loss': 0, 'task_loss': 0, 'domain_loss': 0}
        epoch_source_losses, epoch_target_losses, epoch_domain_losses = [], [], []
        
        # ✅ 修复：分别迭代源域和目标域
        target_iter = iter(target_loader)
        n_batches = len(target_loader)
        
        progress_bar = tqdm(range(n_batches), desc=f'Epoch {epoch+1}/{epochs}', mininterval=3.0)
        
        for batch_idx in progress_bar:
            try:
                # 获取目标域批次
                try:
                    target_batch = next(target_iter)
                except StopIteration:
                    target_iter = iter(target_loader)
                    target_batch = next(target_iter)
                
                # 解析目标域数据
                if len(target_batch) == 4:
                    target_inputs, target_targets, _, target_category = target_batch
                else:
                    target_inputs, target_targets = target_batch[:2]
                    target_category = target_batch[3] if len(target_batch) > 3 else None
                
                target_inputs = target_inputs.float().to(device)
                target_targets = target_targets.float().to(device)
                if target_category is not None:
                    target_category = target_category.float().to(device)
                
                # ✅ 简化：只使用目标域数据进行微调（避免源域数据形状问题）
                if is_adaptive_model:
                    target_predictions = model(target_inputs, target_category, domain_idx=target_domain_idx)
                else:
                    target_predictions = model(target_inputs, target_category)
                
                # 任务损失
                target_task_loss = huber_loss_fn(target_predictions, target_targets)
                
                # 域对抗损失（简化版，仅使用目标域）
                domain_loss = torch.tensor(0.0, device=device)
                
                # 动态权重（随着训练增加目标域权重）
                tgt_weight = min(5.0, 1.0 + epoch*4/epochs)
                
                total_loss = target_task_loss * tgt_weight + domain_loss * current_lambda
                
                # 反向传播
                optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                # 记录
                epoch_source_losses.append(0.0)  # 简化版不使用源域
                epoch_target_losses.append(target_task_loss.item())
                epoch_domain_losses.append(domain_loss.item())
                epoch_stats['total_loss'] += total_loss.item()
                
                progress_bar.set_postfix({
                    'loss': f"{total_loss.item():.4f}",
                    'tgt': f"{target_task_loss.item():.4f}"
                })
                
            except Exception as e:
                logging.warning(f"批次 {batch_idx} 处理出错: {e}")
                tb.print_exc()  # ✅ 使用别名
                continue
        
        scheduler.step()
        
        # 验证
        current_rmse = float('inf')
        with torch.no_grad():
            model.eval()
            val_preds, val_targets = [], []
            
            eval_batches = min(5, len(target_loader))
            eval_iter = iter(target_loader)
            
            for _ in range(eval_batches):
                try:
                    val_batch = next(eval_iter)
                    
                    if len(val_batch) == 4:
                        inputs, targets, _, category = val_batch
                    else:
                        inputs, targets = val_batch[:2]
                        category = val_batch[3] if len(val_batch) > 3 else None
                    
                    inputs = inputs.float().to(device)
                    targets = targets.float().to(device)
                    if category is not None:
                        category = category.float().to(device)
                    
                    if is_adaptive_model:
                        preds = model(inputs, category, domain_idx=target_domain_idx)
                    else:
                        preds = model(inputs, category)
                    
                    val_preds.append(preds)
                    val_targets.append(targets)
                except StopIteration:
                    break
                except Exception as e:
                    logging.warning(f"验证批次出错: {e}")
                    continue
            
            if val_preds:
                all_preds = torch.cat(val_preds, dim=0)
                all_tgts = torch.cat(val_targets, dim=0)
                current_rmse = torch.sqrt(torch.mean((all_preds - all_tgts) ** 2)).item()
                history['rmse'].append(current_rmse)
                
                if current_rmse < best_rmse:
                    best_rmse = current_rmse
                    best_model_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                    logging.info(f"✅ 新最佳RMSE: {best_rmse:.4f}")
                else:
                    patience_counter += 1
        
        avg_loss = epoch_stats['total_loss'] / max(n_batches, 1)
        history['total_loss'].append(avg_loss)
        history['task_loss'].append(sum(epoch_target_losses)/max(len(epoch_target_losses), 1))
        history['domain_loss'].append(sum(epoch_domain_losses)/max(len(epoch_domain_losses), 1))
        
        logging.info(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, RMSE: {current_rmse:.4f}")
        
        if patience_counter >= early_stopping_patience:
            logging.info(f"早停于 epoch {epoch+1}")
            break
    
    if best_model_state:
        model.load_state_dict(best_model_state)
        logging.info(f"已恢复最佳模型 (RMSE: {best_rmse:.4f})")
    
    return model, history

## 使用所有类别数据训练的模型作为源域模型

In [7]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import numpy as np
import traceback
import logging
from datetime import datetime, timezone

# 导入修改后的数据加载器
from dataloader import (
    load_source_domain_dataloaders,
    load_transfer_learning_dataloaders
)


# 导入模型
from bilstm import BaselineBiLSTM
from AdaptiveBILSTM import AdaptiveBiLSTM

# -------------------------- 模型参数设置 --------------------------
batch_size = 32
sequence_length = 24
forecast_horizon = 24
hidden_dim = 64
num_layers = 2
dropout = 0.3
learning_rate = 0.001
weight_decay = 1e-5
epochs = 10
transfer_epochs = 10
input_dim = 6
num_domains = 6

# 定义域索引映射
DOMAIN_MAPPING = {
    "DO": 1,
    "HO": 2,
    "LI": 3,
    "OF": 4,
    "UL": 5,
    "CC": 0  # CC使用默认值0
}

source_model_path = 'models/adaptive_source_huber_best.pth' 
transfer_results = {}
source_domain_idx = 0

# 数据缺失场景
data_shortage_scenarios = ['mild', 'heavy', 'extreme']

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('transfer_learning.log'),
        logging.StreamHandler()
    ]
)

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检查CUDA是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 加载类别信息
with open('train_test_labels.json', 'r') as f:
    train_test_labels = json.load(f)
    categories = list(train_test_labels.keys())

# -------------------------- 加载源域模型 --------------------------
if os.path.exists(source_model_path):
    checkpoint = torch.load(source_model_path)
    
    source_model = AdaptiveBiLSTM(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        category_dim=len(categories),
        forecast_horizon=forecast_horizon,
        num_buildings=1,
        num_domains=num_domains,
        num_layers=num_layers,
        dropout=dropout
    )
    
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        source_model.load_state_dict(checkpoint['state_dict'])
        print(f"✅ 从检查点加载自适应源域模型成功: {source_model_path}")
        print(f"   模型保存于第 {checkpoint.get('epoch', 'unknown')} 轮")
        print(f"   验证损失: {checkpoint.get('val_loss', 'unknown')}")
    else:
        source_model.load_state_dict(checkpoint)
        print(f"✅ 加载自适应源域模型成功: {source_model_path}")
else:
    print(f"❌ 错误：找不到自适应源域模型 {source_model_path}")
    import sys
    sys.exit(1)

# -------------------------- Huber损失基线模型 --------------------------
class HuberBaselineBiLSTM(BaselineBiLSTM):
    """支持直接输出的基线BiLSTM模型，使用Huber损失训练"""
    def __init__(self, input_dim, hidden_dim, forecast_horizon, num_buildings=1, num_layers=2, dropout=0.3):
        super().__init__(input_dim, hidden_dim, forecast_horizon, num_buildings, num_layers, dropout)
        self.fc = torch.nn.Linear(hidden_dim * 2, forecast_horizon)
        
    def forward(self, x, category_onehot=None):
        batch_size, num_buildings, seq_len, _ = x.shape
        x = x.reshape(batch_size * num_buildings, seq_len, -1)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        predictions = self.fc(out)
        predictions = predictions.view(batch_size, num_buildings, -1)
        return predictions

def train_huber_model(model, train_loader, val_loader, epochs=20, lr=0.001, weight_decay=1e-5,device='cuda'):
    """训练使用Huber损失的模型"""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    
    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    huber_loss_fn = HuberLoss(delta=1.0)
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        epoch_loss = 0
        for batch_data in train_loader:
            if len(batch_data) == 4:
                inputs, targets, _, category_onehot = batch_data
            else:
                inputs, targets = batch_data[:2]
                category_onehot = batch_data[3] if len(batch_data) > 3 else None
            
            inputs = inputs.float().to(device)
            targets = targets.float().to(device)
            if category_onehot is not None:
                category_onehot = category_onehot.float().to(device)
            
            optimizer.zero_grad()
            predictions = model(inputs, category_onehot)
            loss = huber_loss_fn(predictions, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            
        train_loss = epoch_loss / len(train_loader)
        history["train_loss"].append(train_loss)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_data in val_loader:
                if len(batch_data) == 4:
                    inputs, targets, _, category_onehot = batch_data
                else:
                    inputs, targets = batch_data[:2]
                    category_onehot = batch_data[3] if len(batch_data) > 3 else None
                
                inputs = inputs.float().to(device)
                targets = targets.float().to(device)
                if category_onehot is not None:
                    category_onehot = category_onehot.float().to(device)
                
                predictions = model(inputs, category_onehot)
                loss = huber_loss_fn(predictions, targets)
                val_loss += loss.item()
                
        val_loss = val_loss / len(val_loader)
        history["val_loss"].append(val_loss)
        scheduler.step(val_loss)
        
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # 早停
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    if best_model_state:
        model.load_state_dict(best_model_state)
        
    return model, history, best_val_loss

# -------------------------- 主迁移学习循环 --------------------------
os.makedirs('models', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

current_user = "leime2ijuan"
source_model_category = 'ALL'
current_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
logging.info(f"Starting transfer learning experiment")
logging.info(f"Time: {current_time}")
logging.info(f"User: {current_user}")

for target_category in categories:
    print(f"\n{'='*70}")
    print(f"当前处理类别: {target_category}")
    print(f"{'='*70}")
    
    transfer_results[target_category] = {}
    target_domain_idx = DOMAIN_MAPPING.get(target_category, 0)
    
    for data_shortage in data_shortage_scenarios:
        logging.info(f"\n🚀 Source(ALL) ➜ Target({target_category}), Shortage: {data_shortage}")
        
        try:
            # ✅ 使用修复版数据加载器
            train_loader, test_loader, target_categories = load_transfer_learning_dataloaders(
                category=target_category,
                data_shortage=data_shortage,
                batch_size=batch_size,
                sequence_length=sequence_length,
                forecast_horizon=forecast_horizon,
                handle_missing='forward_fill'
            )
            
            if train_loader is None or test_loader is None:
                logging.error(f"无法创建数据加载器，跳过 {target_category}/{data_shortage}")
                continue
            
            # ✅ 验证数据形状一致性
            sample = next(iter(train_loader))
            print(f"训练数据形状: 输入={sample[0].shape}, 目标={sample[1].shape}")
            
            # 训练基线模型
            logging.info(f"📦 训练基线模型...")
            baseline_model = HuberBaselineBiLSTM(
                input_dim=input_dim,
                hidden_dim=hidden_dim,
                forecast_horizon=forecast_horizon,
                num_buildings=1,  # ✅ 统一为单建筑
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
            
            # ✅ 修复为
            trained_baseline, baseline_history, best_val_loss = train_huber_model(
                model=baseline_model,
                train_loader=train_loader,
                val_loader=test_loader,
                epochs=transfer_epochs // 2,
                lr=learning_rate,
                weight_decay=weight_decay,
                device=device
            )
            
            # ✅ 域自适应迁移学习（使用修复版函数）
            logging.info(f"🔄 开始迁移学习...")
            adapted_model, transfer_history = adapt_to_target_domain(
                source_model=source_model,
                source_loader=train_loader,  # 混合数据作为源
                target_loader=test_loader,   # 目标域测试数据
                epochs=transfer_epochs,
                lr=learning_rate / 2,
                device=device,
                lambda_domain=0.4,
                early_stopping_patience=5,
                source_domain_idx=source_domain_idx,
                target_domain_idx=target_domain_idx
            )
            
            if adapted_model is None:
                logging.error(f"迁移学习失败")
                continue

            # 评估模型性能
            logging.info(f"📊 评估迁移学习模型性能...")
            metrics = evaluate_model(
                model=adapted_model,
                test_loader=test_loader,
                model_name=f"Adaptive_TL_ALL_to_{target_category}_{data_shortage}",
                baseline_model=trained_baseline,
                device=device,
                domain_idx=target_domain_idx
            )

            # 保存结果
            is_adaptive_model = hasattr(adapted_model, 'time_series_encoder')
            model_type = "adaptive" if is_adaptive_model else "chronos"
            baseline_metrics = metrics.get('baseline_metrics', {})
            
            transfer_results[target_category][data_shortage] = {
                'metrics': metrics,
                'is_adaptive_model': is_adaptive_model,
                'model_type': model_type,
                'transfer_metrics': {
                    'a_distance': metrics.get('a_distance'),
                    'feature_alignment': metrics.get('feature_alignment'),
                    'mmd': metrics.get('mmd'),
                    'is_negative_transfer': metrics.get('is_negative_transfer'),
                    'transfer_gain': metrics.get('transfer_gain'),
                },
                'evaluation_metrics': {
                    'RMSD': metrics.get('RMSD'),
                    'MAPE': metrics.get('MAPE'),
                    'R2': metrics.get('R2'),
                    'CV-RMSE': metrics.get('CV-RMSE'),
                    'CC': metrics.get('CC'),
                    'PICP': metrics.get('PICP'),
                    'NMPIW': metrics.get('NMPIW'),
                },
                'baseline_metrics': baseline_metrics
            }

            # 保存模型
            model_save_path = f'models/{model_type}_TL_ALL_to_{target_category}_{data_shortage}.pth'
            torch.save({
                'state_dict': adapted_model.state_dict(),
                'metrics': metrics,
                'history': transfer_history,
                'target_category': target_category,
                'data_shortage': data_shortage,
            }, model_save_path)
            logging.info(f"✅ 模型已保存到 {model_save_path}")

            # 打印迁移学习指标
            print("\n迁移学习指标:")
            a_distance = metrics.get('a_distance', 'N/A')
            print(f"A-distance: {a_distance:.4f}" if isinstance(a_distance, (int, float)) else f"A-distance: {a_distance}")
            
            mmd = metrics.get('mmd', 'N/A')
            print(f"MMD: {mmd:.4f}" if isinstance(mmd, (int, float)) else f"MMD: {mmd}")
            
            if metrics.get('is_negative_transfer') is not None:
                print(f"负迁移: {'是' if metrics['is_negative_transfer'] else '否'}")

        except Exception as e:
            logging.error(f"❌ 错误: {target_category}/{data_shortage} 迁移失败: {e}")
            traceback.print_exc()
            continue

# 保存所有结果

izable(transfer_results), f, indent=4)
    
    logging.info(f"✅ 所有迁移学习结果已保存到 {results_path}")
except Exception as e:
    logging.error(f"❌ 保存结果时出错: {e}")

print("\n✅ 所有迁移学习实验已完成")

使用设备: cuda


INFO:root:Starting transfer learning experiment
INFO:root:Time: 2026-01-08 16:47:30
INFO:root:User: leime2ijuan
INFO:root:
🚀 Source(ALL) ➜ Target(DO), Shortage: mild


✅ 从检查点加载自适应源域模型成功: models/adaptive_source_huber_best.pth
   模型保存于第 15 轮
   验证损失: 0.0035579158479709235

当前处理类别: DO
📊 加载迁移学习数据 - 类别: DO, 稀缺度: MILD
说明:
  • 训练数据: train标签建筑(完整) + 目标建筑稀缺训练集(前80%)
  • 测试数据: 目标建筑稀缺测试集(后20%)

🏢 建筑分组:
  训练建筑: 5 个 - ['Hog_lodging_Brian', 'Hog_lodging_Nikki', 'Hog_lodging_Ora', 'Robin_lodging_Celia', 'Robin_lodging_Elmer']
  目标建筑: 1 个 - ['Robin_lodging_Renea']

⚡ 加载电力数据...
  ✓ 完整电力数据: (17544, 25)
  ✓ 稀缺电力数据: (17544, 6)
  ✓ 稀缺训练集: (14035, 6) (前80%)
  ✓ 稀缺测试集: (3509, 6) (后20%)

🌤️  加载天气数据...
  训练建筑天气数据 (完整):
    ✓ Hog: (17542, 5)
    ✓ Robin: (17516, 5)
  目标建筑天气数据 (mild):
    ✓ Robin: (17516, 5) (缺失率: 16.02%)

🔄 创建数据加载器...
  创建训练建筑数据集 (5 个建筑)...
  数据集: 5 个建筑, 17462 个有效样本
  创建目标建筑训练集数据集 (1 个建筑)...
  ⚠️  数据包含缺失 (缺失率: 20.00%)
  数据集: 1 个建筑, 13962 个有效样本
  创建目标建筑测试集数据集 (1 个建筑)...
  数据集: 1 个建筑, 3456 个有效样本

✅ 类别 DO 迁移学习数据加载完成!
训练样本: 31424
  - 训练建筑数据: 17462 样本
  - 目标建筑训练数据: 13962 样本
测试样本: 3456
批次大小: 32
缺失处理: forward_fill



INFO:root:📦 训练基线模型...


训练数据形状: 输入=torch.Size([92, 1, 24, 6]), 目标=torch.Size([92, 1, 24])
Epoch 1/5, Train Loss: 0.0067, Val Loss: 0.0036
Epoch 2/5, Train Loss: 0.0033, Val Loss: 0.0033
Epoch 3/5, Train Loss: 0.0030, Val Loss: 0.0030
Epoch 4/5, Train Loss: 0.0028, Val Loss: 0.0030


...

INFO:root:Epoch 10/10 - Loss: 0.0315, RMSE: 0.1183
INFO:root:早停于 epoch 10
INFO:root:已恢复最佳模型 (RMSE: 0.1092)
INFO:root:📊 评估迁移学习模型性能...


Evaluating Adaptive_TL_ALL_to_CC_extreme:   0%|          | 0/109 [00:00<?, ?it/s]


数据形状:
预测: (83064,), 目标: (83064,)
下界: (83064,), 上界: (83064,)


INFO:root:✅ 模型已保存到 models/adaptive_TL_ALL_to_CC_extreme.pth
INFO:root:✅ 所有迁移学习结果已保存到 models/transfer_learning_results_20260109_203221.json


警告: 基线模型的迁移学习数据缺失，跳过相关指标

Adaptive_TL_ALL_to_CC_extreme 评估报告:
RMSD: 0.1153
MAPE: 30.72%
R²: 0.5229
CV-RMSE: 24.6188%
SD_real: 0.1670
SD_pred: 0.0982
CC: 0.7444
RMSD改进率(PIR): 8.64%

Baseline Model                   评估报告:
RMSD: 0.1263
MAPE: 35.78%
R²: 0.4283
CV-RMSE: 26.9481%
SD_real: 0.1670
SD_pred: 0.0980
CC: 0.6641

基线模型不确定性评估:
PICP: 35.93%
NMPIW: 0.12913177907466888
校准误差: 59.07%
平均不确定性: 0.10000001639127731

迁移学习评估:
A-distance: 0.8685
特征对齐: 0.6607
MMD: 0.2755

不确定性评估:
PICP: 42.88%
NMPIW: 0.14061851799488068
校准误差: 52.12%
UQS: 0.2868
平均不确定性: 0.026795247569680214

迁移学习指标:
A-distance: 0.8685351610183716
MMD: 0.2754802703857422
负迁移: 否

✅ 所有迁移学习实验已完成


In [8]:
# -------------------------- 新增：保存结果到CSV --------------------------
import csv
from datetime import datetime
import numpy as np
import torch
import os

def convert_to_basic_type(obj):
    """将对象转换为基本数据类型，以便保存到CSV"""
    if obj is None:
        return "None"
    elif isinstance(obj, (np.ndarray, torch.Tensor)):
        # 处理数组和张量
        if obj.size == 1:
            try:
                return float(obj.item()) if np.issubdtype(obj.dtype, np.floating) else int(obj.item())
            except:
                return float(obj) if np.issubdtype(type(obj), np.floating) else int(obj)
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64, np.int32, np.int64)):
        # 处理NumPy标量
        return float(obj) if np.issubdtype(obj.dtype, np.floating) else int(obj)
    elif isinstance(obj, (float, int, str, bool)):
        # 基本类型直接返回
        return obj
    else:
        # 其他类型转换为字符串
        return str(obj)

# 确保结果目录存在
os.makedirs('results', exist_ok=True)

# 定义CSV文件路径，添加时间戳避免覆盖
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"results/transfer_results_{timestamp}.csv"

# 准备CSV表头（简化版，按模型类型分类）
csv_header = [
    "Target Category", "Data Shortage", "Model Type", 
    "RMSD", "MAPE (%)", "R2", "CV-RMSE (%)", "SD_real", "SD_pred", "CC", 
    "PIR (%)", "PICP (%)", "NMPIW", "Calibration Error (%)", "UQS", "Avg Uncertainty",
    "A-distance", "Feature Alignment", "MMD", "Negative Transfer", "Transfer Gain", "Sample Efficiency",
    "Baseline A-distance", "Baseline Feature Alignment", "Baseline MMD", "RMSD Improvement (%)"
]

# 准备元数据行
metadata_rows = [
    ["Transfer Learning Experiment Results"],
    ["Date", datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    ["Source Model", source_model_category],
    ["User", current_user],
    [""],  # 空行分隔
]

# 准备数据行
csv_rows = []

# 按类别分类的结果
category_results = {category: [] for category in categories}

# 汇总改进率数据（用于热力图）
improvement_data = np.zeros((len(categories), len(data_shortage_scenarios)))
improvement_data.fill(np.nan)  # 初始化为NaN

# 遍历所有实验结果
for i, target_category in enumerate(categories):
    if target_category not in transfer_results:
        continue
        
    for j, data_shortage in enumerate(data_shortage_scenarios):
        if data_shortage not in transfer_results[target_category]:
            continue
            
        result = transfer_results[target_category][data_shortage]
        
        # 处理基线模型
        baseline_metrics = result.get('baseline_metrics', {})
        if baseline_metrics:
            # 获取基线模型的迁移指标
            baseline_transfer_metrics = baseline_metrics.get('transfer_metrics', {})
            
            row = [
                target_category, data_shortage, "Baseline",
                convert_to_basic_type(baseline_metrics.get('RMSD', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('MAPE', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('R2', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('CV-RMSE', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('SD_real', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('SD_pred', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('CC', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('PIR', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('PICP', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('NMPIW', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('calibration_error', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('UQS', 'N/A')),  # 可能不存在，返回N/A
                convert_to_basic_type(baseline_metrics.get('avg_uncertainty', 'N/A')),
                "N/A", "N/A", "N/A", "N/A", "N/A", "N/A",  # 迁移模型特有指标
                # 基线模型的迁移指标
                convert_to_basic_type(baseline_transfer_metrics.get('a_distance', 'N/A')),
                convert_to_basic_type(baseline_transfer_metrics.get('feature_alignment', 'N/A')),
                convert_to_basic_type(baseline_transfer_metrics.get('mmd', 'N/A')),
                "N/A"  # 改进率（基线模型不适用）
            ]
            csv_rows.append(row)
            category_results[target_category].append(row)
        
        # 处理迁移模型
        model_metrics = result.get('evaluation_metrics', {})
        if not model_metrics:
            model_metrics = result.get('metrics', {})  # 兼容旧格式
            
        transfer_metrics = result.get('transfer_metrics', {})
        
        if model_metrics:
            # 计算RMSD改进率
            baseline_rmsd = convert_to_basic_type(baseline_metrics.get('RMSD', 'N/A'))
            transfer_rmsd = convert_to_basic_type(model_metrics.get('RMSD', 'N/A'))
            
            if isinstance(baseline_rmsd, (int, float)) and isinstance(transfer_rmsd, (int, float)) and baseline_rmsd > 0:
                rmsd_improvement = (baseline_rmsd - transfer_rmsd) / baseline_rmsd * 100
                # 保存到热力图数据
                improvement_data[i, j] = rmsd_improvement
            else:
                rmsd_improvement = 'N/A'
            
            row = [
                target_category, data_shortage, result.get('model_type', 'Transfer'),
                convert_to_basic_type(model_metrics.get('RMSD', 'N/A')),
                convert_to_basic_type(model_metrics.get('MAPE', 'N/A')),
                convert_to_basic_type(model_metrics.get('R2', 'N/A')),
                convert_to_basic_type(model_metrics.get('CV-RMSE', 'N/A')),
                convert_to_basic_type(model_metrics.get('SD_real', 'N/A')),
                convert_to_basic_type(model_metrics.get('SD_pred', 'N/A')),
                convert_to_basic_type(model_metrics.get('CC', 'N/A')),
                convert_to_basic_type(model_metrics.get('PIR', 'N/A')),
                convert_to_basic_type(model_metrics.get('PICP', 'N/A')),
                convert_to_basic_type(model_metrics.get('NMPIW', 'N/A')),
                convert_to_basic_type(model_metrics.get('calibration_error', 'N/A')),
                convert_to_basic_type(model_metrics.get('UQS', 'N/A')),
                convert_to_basic_type(model_metrics.get('avg_uncertainty', 'N/A')),
                # 迁移指标
                convert_to_basic_type(transfer_metrics.get('a_distance', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('feature_alignment', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('mmd', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('is_negative_transfer', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('transfer_gain', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('sample_efficiency', 'N/A')),
                # 迁移模型的基线指标（重复列，用于对比）
                convert_to_basic_type(transfer_metrics.get('baseline_a_distance', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('baseline_feature_alignment', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('baseline_mmd', 'N/A')),
                # 改进率
                convert_to_basic_type(rmsd_improvement)
            ]
            csv_rows.append(row)
            category_results[target_category].append(row)

# 写入主CSV文件
try:
    with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        # 写入元数据
        for row in metadata_rows:
            writer.writerow(row)
        # 写入数据表头和内容
        writer.writerow(csv_header)
        writer.writerows(csv_rows)
    print(f"\n💾 迁移学习结果已保存至 {csv_path}")
except Exception as e:
    print(f"\n❌ 保存CSV文件失败: {str(e)}")
    logging.error(f"保存CSV文件失败: {str(e)}")

# 按类别分别保存CSV
for category in categories:
    if category in category_results and category_results[category]:
        try:
            category_csv_path = f"results/transfer_results_{category}_{timestamp}.csv"
            with open(category_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
                writer = csv.writer(csvfile)
                # 写入元数据
                for row in metadata_rows:
                    writer.writerow(row)
                # 添加类别特定信息
                writer.writerow([f"Category: {category}"])
                writer.writerow([""])  # 空行
                # 写入数据表头和内容
                writer.writerow(csv_header)
                writer.writerows(category_results[category])
            print(f"  - 类别 {category} 结果已保存至 {category_csv_path}")
        except Exception as e:
            printf("  ❌ 保存类别 {category} ")


💾 迁移学习结果已保存至 results/transfer_results_20260109_203221.csv
  - 类别 DO 结果已保存至 results/transfer_results_DO_20260109_203221.csv
  - 类别 HO 结果已保存至 results/transfer_results_HO_20260109_203221.csv
  - 类别 LI 结果已保存至 results/transfer_results_LI_20260109_203221.csv
  - 类别 OF 结果已保存至 results/transfer_results_OF_20260109_203221.csv
  - 类别 UL 结果已保存至 results/transfer_results_UL_20260109_203221.csv
  - 类别 CC 结果已保存至 results/transfer_results_CC_20260109_203221.csv


In [9]:
# -------------------------- 新增：保存结果到CSV --------------------------
import csv
from datetime import datetime
import numpy as np
import torch

def convert_to_basic_type(obj):
    """将对象转换为基本数据类型，以便保存到CSV"""
    if obj is None:
        return "None"
    elif isinstance(obj, (np.ndarray, torch.Tensor)):
        # 处理数组和张量
        if obj.size == 1:
            try:
                return float(obj.item()) if np.issubdtype(obj.dtype, np.floating) else int(obj.item())
            except:
                return float(obj) if np.issubdtype(type(obj), np.floating) else int(obj)
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64, np.int32, np.int64)):
        # 处理NumPy标量
        return float(obj) if np.issubdtype(obj.dtype, np.floating) else int(obj)
    elif isinstance(obj, (float, int, str, bool)):
        # 基本类型直接返回
        return obj
    else:
        # 其他类型转换为字符串
        return str(obj)

# 定义CSV文件路径
csv_path = "transfer_results_CC.csv"

# 准备CSV表头（简化版，按模型类型分类）
csv_header = [
    "Target Category", "Data Shortage", "Model Type", 
    "RMSD", "MAPE (%)", "R2", "CV-RMSE (%)", "SD_real", "SD_pred", "CC", 
    "PIR (%)", "PICP (%)", "NMPIW", "Calibration Error (%)", "UQS", "Avg Uncertainty",
    "A-distance", "Feature Alignment", "MMD", "Negative Transfer", "Transfer Gain", "Sample Efficiency",
    "Baseline A-distance", "Baseline Feature Alignment", "Baseline MMD"
]

# 准备数据行
csv_rows = []

# 遍历所有实验结果
for target_category in transfer_results:
    for data_shortage in transfer_results[target_category]:
        result = transfer_results[target_category][data_shortage]
        
        # 处理基线模型
        baseline_metrics = result.get('baseline_metrics', {})
        if baseline_metrics:
            # 获取基线模型的迁移指标
            baseline_transfer_metrics = baseline_metrics.get('transfer_metrics', {})
            
            row = [
                target_category, data_shortage, "Baseline",
                convert_to_basic_type(baseline_metrics.get('RMSD', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('MAPE', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('R2', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('CV-RMSE', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('SD_real', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('SD_pred', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('CC', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('PIR', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('PICP', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('NMPIW', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('calibration_error', 'N/A')),
                convert_to_basic_type(baseline_metrics.get('UQS', 'N/A')),  # 可能不存在，返回N/A
                convert_to_basic_type(baseline_metrics.get('avg_uncertainty', 'N/A')),
                "N/A", "N/A", "N/A", "N/A", "N/A", "N/A",  # 迁移模型特有指标
                # 基线模型的迁移指标
                convert_to_basic_type(baseline_transfer_metrics.get('a_distance', 'N/A')),
                convert_to_basic_type(baseline_transfer_metrics.get('feature_alignment', 'N/A')),
                convert_to_basic_type(baseline_transfer_metrics.get('mmd', 'N/A'))
            ]
            csv_rows.append(row)
        
        # 处理迁移模型
        model_metrics = result.get('metrics', {})
        transfer_metrics = result.get('transfer_metrics', {})
        if model_metrics:
            row = [
                target_category, data_shortage, result.get('model_type', 'Transfer'),
                convert_to_basic_type(model_metrics.get('RMSD', 'N/A')),
                convert_to_basic_type(model_metrics.get('MAPE', 'N/A')),
                convert_to_basic_type(model_metrics.get('R2', 'N/A')),
                convert_to_basic_type(model_metrics.get('CV-RMSE', 'N/A')),
                convert_to_basic_type(model_metrics.get('SD_real', 'N/A')),
                convert_to_basic_type(model_metrics.get('SD_pred', 'N/A')),
                convert_to_basic_type(model_metrics.get('CC', 'N/A')),
                convert_to_basic_type(model_metrics.get('PIR', 'N/A')),
                convert_to_basic_type(model_metrics.get('PICP', 'N/A')),
                convert_to_basic_type(model_metrics.get('NMPIW', 'N/A')),
                convert_to_basic_type(model_metrics.get('calibration_error', 'N/A')),
                convert_to_basic_type(model_metrics.get('UQS', 'N/A')),
                convert_to_basic_type(model_metrics.get('avg_uncertainty', 'N/A')),
                # 迁移指标
                convert_to_basic_type(transfer_metrics.get('a_distance', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('feature_alignment', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('mmd', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('is_negative_transfer', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('transfer_gain', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('sample_efficiency', 'N/A')),
                # 迁移模型的基线指标（重复列，用于对比）
                convert_to_basic_type(transfer_metrics.get('baseline_a_distance', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('baseline_feature_alignment', 'N/A')),
                convert_to_basic_type(transfer_metrics.get('baseline_mmd', 'N/A'))
            ]
            csv_rows.append(row)

# 写入CSV文件
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(csv_header)
    writer.writerows(csv_rows)

print(f"\n💾 迁移学习结果已保存至 {csv_path}")
print("\n📊 Generating additional visualization charts...")


💾 迁移学习结果已保存至 transfer_results_CC.csv

📊 Generating additional visualization charts...
